In [34]:
import random
import re

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

DATA_FILE = "Tweets.csv"

In [35]:
def clean(text):
    text = str(text).lower()
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z' ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def load():
    df = pd.read_csv(DATA_FILE)[["text", "airline_sentiment"]].dropna()
    df["clean"] = df["text"].apply(clean)
    return df


def add_noise(text, rate=0.15, seed=0):
    rnd = random.Random(seed)
    out = []
    for w in text.split():
        if len(w) > 3 and rnd.random() < rate:
            i = rnd.randrange(len(w))
            w = w[:i] + w[i + 1:]
        out.append(w)
    return " ".join(out)

In [36]:
df = load()
print(df.shape)
print(df["airline_sentiment"].value_counts())
print(df["clean"].head())

(14640, 3)
airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64
0                                            what said
1    plus you've added commercials to the experienc...
2    i didn't today must mean i need to take anothe...
3    it's really aggressive to blast obnoxious ente...
4             and it's a really big bad thing about it
Name: clean, dtype: object


In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["airline_sentiment"],
    test_size=0.2, random_state=42, stratify=df["airline_sentiment"])

baseline = y_test.value_counts(normalize=True).max()

model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2),
    LogisticRegression(max_iter=1000, class_weight="balanced"))
model.fit(X_train, y_train)
pred = model.predict(X_test)

print(f"Baseline (always guess the most common): {baseline:.1%}")
print(f"Model accuracy: {accuracy_score(y_test, pred):.1%}")
print(classification_report(y_test, pred))

Baseline (always guess the most common): 62.7%
Model accuracy: 78.8%
              precision    recall  f1-score   support

    negative       0.89      0.85      0.87      1835
     neutral       0.59      0.69      0.63       620
    positive       0.73      0.69      0.71       473

    accuracy                           0.79      2928
   macro avg       0.73      0.74      0.74      2928
weighted avg       0.80      0.79      0.79      2928



In [38]:
noisy = X_test.apply(add_noise)
print(f"Accuracy with typos added: {accuracy_score(y_test, model.predict(noisy)):.1%}")


def confidence_table(model, X_test, y_test):
    proba = model.predict_proba(X_test)
    conf = proba.max(axis=1)
    pred = model.classes_[proba.argmax(axis=1)]
    print("\nThreshold | auto-handled | accuracy on those")
    for t in [0.5, 0.6, 0.7, 0.8, 0.9]:
        mask = conf >= t
        if mask.sum() == 0:
            print(f"{t:.1f}       | 0%           | n/a")
            continue
        acc = (pred[mask] == y_test.values[mask]).mean()
        print(f"{t:.1f}       | {mask.mean():.0%}          | {acc:.1%}")


def escalation_queue(model, df, X_test, top=10):
    proba = model.predict_proba(X_test)
    neg = list(model.classes_).index("negative")
    out = pd.DataFrame({
        "text": df.loc[X_test.index, "clean"].values,
        "negative_score": proba[:, neg],
    })
    out = out.sort_values("negative_score", ascending=False).head(top)
    print("\nTop unhappy-customer messages (escalate first):")
    for _, r in out.iterrows():
        print(f"{r['negative_score']:.2f} | {r['text'][:90]}")


confidence_table(model, X_test, y_test)
escalation_queue(model, df, X_test)

Accuracy with typos added: 78.8%

Threshold | auto-handled | accuracy on those
0.5       | 83%          | 84.1%
0.6       | 65%          | 89.2%
0.7       | 48%          | 93.0%
0.8       | 32%          | 95.4%
0.9       | 14%          | 97.9%

Top unhappy-customer messages (escalate first):
0.99 | been waiting for minutes on hold because yall cancelled flightled my return flight answer 
0.99 | worst experience with you cancelled flightled flight no voucher and no luggage because ram
0.99 | now on hold for hrs min this is ridiculous
0.99 | being delayed hours for a one hour flight is more than frustrating i'm also unable to get 
0.99 | on hold for hour and min trying to find the luggage flight cancelled flightled where is lu
0.99 | customer service is the worst hanging up on customers intentionally after one hour on hold
0.99 | i fucking hate your customer service i've been on hold all morning trying to figure out wh
0.99 | worst customer service still not providing me an answer even a

In [39]:
joblib.dump(model, "sentiment_model.joblib")

THRESHOLD = 0.8


def check(text):
    proba = model.predict_proba([clean(text)])[0]
    label = model.classes_[proba.argmax()]
    conf = proba.max()
    note = "" if conf >= THRESHOLD else "  -> low confidence: send to a human"
    print(f"{text!r} -> {label} ({conf:.0%}){note}")


for msg in [
    "Where to get my boarding pass? ",
    "Idk why I chose this airline.",
    "Can you please tell me where to find the lounge?",
]:
    check(msg)

'Where to get my boarding pass? ' -> neutral (57%)  -> low confidence: send to a human
'Idk why I chose this airline.' -> negative (60%)  -> low confidence: send to a human
'Can you please tell me where to find the lounge?' -> neutral (70%)  -> low confidence: send to a human
